In [1]:
from pinecone import Pinecone
from openai import OpenAI
from google import genai

import pandas as pd
import json
import os

In [2]:
with open('../cfg.json', 'r') as f:
    config = json.load(f)

In [3]:
pc = Pinecone(api_key=config['pinecone_api_key'])

dense_index_name = "10k-dense"
sparse_index_name = "10k-sparse"

dense_index = pc.Index(dense_index_name)
sparse_index = pc.Index(sparse_index_name)

c:\Users\berca\anaconda3\envs\rag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
client = genai.Client(api_key=config['google_genai_api_key'])

In [5]:
def chat(query: str, dense_index, sparse_index, genai_client, log_path) -> str:
    
    tickers = []
    query_lower = query.lower()
    
    if any(keyword in query_lower for keyword in ['apple', 'aapl', 'ipad', 'iphone', 'macbook', 'airpods']):
        tickers.append('AAPL')
    
    if any(keyword in query_lower for keyword in ['microsoft', 'msft', 'windows', 'xbox', 'surface', 'azure', 'office']):
        tickers.append('MSFT')
    
    if any(keyword in query_lower for keyword in ['amazon', 'amzn', 'aws', 'alexa']):
        tickers.append('AMZN')
    
    if any(keyword in query_lower for keyword in ['google', 'googl', 'alphabet', 'android', 'youtube', 'chrome']):
        tickers.append('GOOGL')
    
    if any(keyword in query_lower for keyword in ['nvidia', 'nvda', 'gpu', 'cuda', 'geforce']):
        tickers.append('NVDA')

    # don't search all the tickers if we know which one(s) the user is asking about
    if len(tickers) > 0:
        results = dense_index.search(
            namespace="namespace",
            query={
                "top_k": 20,
                "inputs": {
                    "text": query
                },
                "filter": {
                    "ticker": {"$in": tickers}
                }
            }
        )
    else:
        results = dense_index.search(
            namespace="namespace",
            query={
                "top_k": 30,
                "inputs": {
                    "text": query
                }
            },
            fields=["chunk_text", "ticker", "fiscal_year"]
        )

    # Extract only text from results
    text_results = "\n\n".join([f"""This information is about {match["fields"]["ticker"]}'s {match["fields"]["fiscal_year"]} financial statements:\n{match["fields"]["chunk_text"]}""" for match in results["result"]['hits']])

    response = genai_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=f"""Use this information:
            {text_results}
            to answer the question: {query}""",
        config=genai.types.GenerateContentConfig(
            system_instruction="You're a financial expert. You work as an assistant to financial analysts. You need to support your claims with direct quotes from the provided information.",
            temperature=0.5
        )
    )
    
    llm_answer = response.text
    
    # Create dataframe with query, retriever results, and LLM answer
    df = pd.DataFrame({
        "timestamp": [pd.Timestamp.now()],
        "query": [query],
        "tickers": [", ".join(tickers)],
        "retriever_results": [text_results],
        "llm_answer": [llm_answer]
    })

    df.to_csv(log_path, mode='a', header=not os.path.exists(log_path), index=False)

    return llm_answer

In [6]:
# print(chat("What was Apple's revenue from the iPad product line?", dense_index, sparse_index, client, '../log.csv'))
# print(chat("Compare Amazon's and Nvidia's operating risks.", dense_index, sparse_index, client, '../log.csv'))
print(chat("Summarize Google sales for 2024.", dense_index, sparse_index, client, '../log.csv'))

In 2024, Google's total revenues reached $350,018 million. This was comprised of $304,930 million from Google Services, $43,229 million from Google Cloud, and $1,648 million from Other Bets. Additionally, there were hedging gains of $211 million.

Geographically, revenue was distributed as follows:
*   United States: $170,447 million (49%)
*   EMEA: $102,127 million (29%)
*   APAC: $56,815 million (16%)
*   Other Americas: $20,418 million (6%)
